# Housing Strand - Evidence Dashboard Workbench

**MultimodalAI'26 - Housing Demo**

This notebook builds all five required Evidence Dashboard views as starter evidence for teams.

It does **not** produce final submission deliverables for participants.

## Objective

Produce explicit outputs for all required views:
1. Sensor data quality
2. MNAR analysis (at least two patterns)
3. Subgroup equity
4. Leakage audit
5. Dataset audit verdicts

**Three-role reminder**
- Builder: connects these findings to solution behavior.
- Evidence Analyst: computes and validates the views.
- Governance Lead: writes final benchmark/pathway verdict narratives.

**Prerequisite**
- Run notebooks 01, 02, and 03 first.

**Important**
- Final `omaib_pathway.json` and `housing_benchmark_card.json` must be completed by participants in root `reference/`.

## Section 1 - Setup and Load Inputs

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    DEMO_ROOT = cwd
elif (cwd / "demo" / "src").exists():
    DEMO_ROOT = cwd / "demo"
else:
    DEMO_ROOT = cwd.parent

sys.path.append(str(DEMO_ROOT))

from src.evidence import (
    aggregate_dataset_audit,
    compute_equity_view,
    compute_leakage_view,
    compute_mnar_view,
    compute_sensor_quality_view,
)

PROCESSED_DIR = DEMO_ROOT / "data" / "processed"
METRICS_DIR = DEMO_ROOT / "saved_metrics"

merged = pd.read_csv(PROCESSED_DIR / "merged_demo.csv")
train_df = pd.read_csv(PROCESSED_DIR / "train_features.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test_features.csv")
equity_df = pd.read_csv(METRICS_DIR / "subgroup_equity.csv")
model_payload = json.loads((METRICS_DIR / "model_results.json").read_text(encoding="utf-8"))
threshold_counts = model_payload["threshold_counts"]

merged["cold_risk"] = (merged["avgTemperature"] < 19.0).astype(int)
merged.head()

## Section 2 - View 1: Sensor Data Quality

In [ ]:
sensor_quality_view = compute_sensor_quality_view(merged)
sensor_quality_view

### Visual check: missingness by modality

This plot helps the Evidence Analyst communicate sensor quality findings quickly to the team and Governance Lead.

missing = pd.Series(sensor_quality_view["missingness_rates"]).sort_values(ascending=False)
plt.figure(figsize=(8, 3))
sns.barplot(x=missing.index, y=missing.values)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Missing rate")
plt.title("Sensor Missingness by Modality")
plt.tight_layout()
plt.show()

## Section 3 - View 2: MNAR Analysis (at least two patterns)

In [ ]:
mnar_view = compute_mnar_view(merged)
mnar_view

## Section 4 - View 3: Subgroup Equity

In [ ]:
equity_rows = equity_df.to_dict(orient="records")
equity_view = compute_equity_view(equity_rows)
display(equity_df)
equity_view

### Visual check: subgroup AUROC by model/group

Use this to discuss subgroup behavior across your model set before writing equity verdicts.

In [ ]:
plt.figure(figsize=(8, 3))
sns.barplot(data=equity_df, x="model", y="auroc", hue="group")
plt.xticks(rotation=20, ha="right")
plt.ylabel("AUROC")
plt.title("Subgroup AUROC by Model")
plt.tight_layout()
plt.show()

## Section 5 - View 4: Leakage Audit

In [ ]:
leakage_view = compute_leakage_view(train_df, test_df)
leakage_view

## Section 6 - View 5: Dataset Audit Verdicts

In [ ]:
dataset_audit = aggregate_dataset_audit(
    sensor_quality_verdict=sensor_quality_view["verdict"],
    split_integrity_verdict=leakage_view["verdict"],
    equity_verdict=equity_view["verdict"],
)

evidence_dashboard = {
    "sensor_data_quality": sensor_quality_view,
    "mnar_analysis": mnar_view,
    "subgroup_equity": equity_view,
    "leakage_audit": leakage_view,
    "dataset_audit": {
        "data_quality": dataset_audit.data_quality,
        "split_integrity": dataset_audit.split_integrity,
        "equity": dataset_audit.equity,
        "overall_verdict": dataset_audit.overall,
    },
}

evidence_dashboard

## Section 7 - Save Evidence Artifacts (Starter Only)

This section stores analysis artifacts for team discussion.

Participants must manually complete root `reference/` deliverables.

In [ ]:
thresholds = [0.3, 0.5, 0.7]
flagged_counts = [int(threshold_counts[str(t)]) for t in thresholds]

evidence_dashboard["threshold_sensitivity_preview"] = {
    "thresholds_tested": thresholds,
    "flagged_count_at_each_threshold": flagged_counts,
}

(METRICS_DIR / "evidence_dashboard.json").write_text(json.dumps(evidence_dashboard, indent=2), encoding="utf-8")

print("Saved: ", METRICS_DIR / "evidence_dashboard.json")
print("Next step for participants:")
print("- Fill root reference/omaib_pathway.json")
print("- Fill root reference/housing_benchmark_card.json")
print("- Complete option_specific content with your team findings")